In [1]:
import random
from typing import Annotated
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.messages import ToolMessage


In [2]:
SECRET_NUMBER = random.randint(0, 100)
print(f"Secret number is: {SECRET_NUMBER}")

Secret number is: 57


In [3]:
@tool
def evaluate_guess(guess: int) -> str:
    """Evaluates the user's guess against the secret number.
    Returns feedback: 'too high', 'too low', or 'correct'.
    """
    if guess == SECRET_NUMBER:
        return f"Correct! The number was {SECRET_NUMBER}. You won!"
    elif guess < SECRET_NUMBER:
        return "Too low! Try a higher number."
    else:
        return "Too high! Try a lower number."

In [4]:
class GameState(TypedDict):
    """
    This defines what information the agent tracks during the game.
    Think of it as the agent's "memory" or "workspace".
    """
    messages: Annotated[list, add_messages]
    game_won: bool
    guess_count: int

In [5]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [6]:
tools = [evaluate_guess]
llm_with_tools = llm.bind_tools(tools)

In [7]:
from langchain_core.messages import ToolMessage

SYSTEM_PROMPT = """You are a helpful number guessing game guide.
A secret number is between 0 and 100.
The user will make guesses, and you will:
1. Use the evaluate_guess tool to check each guess
2. Give encouraging feedback
3. Remember previous guesses and guide them

Be friendly and helpful. Keep track of the range (high/low bounds)."""

def agent_node(state: GameState):
    messages = state["messages"]
    messages_with_system = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    response = llm_with_tools.invoke(messages_with_system)
    return {
        "messages": [response],
        "game_won": state["game_won"],
        "guess_count": state["guess_count"]
    }

def tool_node(state: GameState):
    messages = state["messages"]
    last_message = messages[-1]
    tool_calls = last_message.tool_calls
    
    tool_results = []
    for tool_call in tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        if tool_name == "evaluate_guess":
            result = evaluate_guess.invoke(tool_args)
            guess_count = state["guess_count"] + 1
            game_won = "Correct!" in result
            
            # FIX: Use ToolMessage instead of dict
            tool_results.append(
                ToolMessage(content=result, tool_call_id=tool_call["id"])
            )
            
            return {
                "messages": tool_results,
                "game_won": game_won,
                "guess_count": guess_count
            }
    
    return state

In [8]:
graph = StateGraph(GameState)

graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")


In [9]:
def should_continue(state: GameState):
    """
    Decides whether to continue the game or stop.
    """
    if state["game_won"]:
        return END
    
    messages = state["messages"]
    
    # FIX: Check if messages is empty
    if not messages:
        return "agent"
    
    last_message = messages[-1]
    
    # FIX: Check if tool_calls exists
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    else:
        return "agent"

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "agent": "agent",
        END: END
    }
)

graph.add_edge("tools", "agent")

app = graph.compile()

In [ ]:
import copy

initial_state = {
    "messages": [],
    "game_won": False,
    "guess_count": 0
}

def play_game():
    state = copy.deepcopy(initial_state)  # FIX: Deep copy
    
    print("\n🎮 Welcome to the Number Guessing Game!")
    print("Think of a number between 0 and 100.")
    print("Type 'quit' to exit.\n")
    
    while not state["game_won"]:
        user_input = input("Your guess: ").strip()
        
        if user_input.lower() == "quit":
            print("Thanks for playing!")
            break
        
        try:
            guess = int(user_input)
            if guess < 0 or guess > 100:
                print("❌ Please enter a number between 0 and 100.\n")
                continue
        except ValueError:
            print("❌ Please enter a valid number.\n")
            continue
        
        user_message = HumanMessage(content=f"I guess {guess}")
        state["messages"].append(user_message)
        
        print("\n⏳ Agent thinking...\n")
        state = app.invoke(state)
        
        last_message = state["messages"][-1]
        if hasattr(last_message, "content"):
            print(f"Agent: {last_message.content}\n")
        
        print(f"Guesses so far: {state['guess_count']}\n")
        
        if state["game_won"]:
            print(f"\n🎉 Game Over! You won in {state['guess_count']} guesses!")
            break

if __name__ == "__main__":
    play_game()


🎮 Welcome to the Number Guessing Game!
Think of a number between 0 and 100.
Type 'quit' to exit.

